# **Project Name** — Voyage Analytics — Flight Price Regression

##### **Project Type** — EDA / Regression
##### **Contribution** — Individual
##### **Team Member 1 — Harsh Paul**

# **Project Summary**

This notebook predicts a flight price from quote-time characteristics:
origin, destination, ticket type, agency, duration, distance, and booking date.
The result can support travel budgeting, quote review, and pricing analysis.

This standalone notebook uses the supplied travel_capstone data, follows the
provided EDA submission template, and is ready to run in Google Colab. It explains
the data checks, modelling choices, evaluation approach, and business meaning of
the findings rather than only printing code output.

# **GitHub Link**

[https://github.com/harshpaul1999-tech/Voyage-Analytics](https://github.com/harshpaul1999-tech/Voyage-Analytics)

# **Problem Statement**

#### **Define Your Business Objective?**

Build a price-estimation model for a new flight quote. The evaluation
should imitate future deployment, so the model is trained on earlier bookings and
tested on the later time period.

All model inputs are limited to information available at prediction time. The
holdout logic is kept separate from fitting so reported results reflect unseen
examples rather than memorisation.

# **General Guidelines**

- Run every cell from top to bottom in a fresh Colab runtime.
- The setup cell clones the public GitHub repository automatically in Colab.
- The charts and metrics are calculated from the supplied data, not typed in.
- A fixed random seed is used when a split or model needs randomness.

# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
from pathlib import Path
import sys
if "google.colab" in sys.modules:
    if not Path("/content/Voyage-Analytics").exists():
        !git clone --depth 1 https://github.com/harshpaul1999-tech/Voyage-Analytics.git /content/Voyage-Analytics
    %cd /content/Voyage-Analytics

import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

sns.set_theme(style="whitegrid", palette="deep")
RANDOM_STATE = 42
DATA_DIR = Path("data/raw")

### Dataset Loading

In [ ]:
flights = pd.read_csv(DATA_DIR / "flights.csv")
flights["date"] = pd.to_datetime(flights["date"], format="%m/%d/%Y", errors="coerce")
print("Loaded flights:", flights.shape)

### Dataset First View

In [ ]:
display(flights.head())

### Dataset Rows & Columns count

In [ ]:
print(flights.shape)

### Dataset Information

In [ ]:
flights.info()

#### Duplicate Values

In [ ]:
print('Duplicate rows:', flights.duplicated().sum())

#### Missing Values/Null Values

In [ ]:
display(flights.isna().sum().sort_values(ascending=False).to_frame('missing_values'))

### What did you know about your dataset?

Each row is a flight record. Price is the supervised-learning target.
The date provides the ordering required for a time-aware test set. Travel and user
codes are identifiers, so they are excluded from the predictive feature set.

## ***2. Understanding Your Variables***

### Variables Description

| Variable | Meaning | Model role |
|---|---|---|
| price | Flight price | Regression target |
| from, to | Route endpoints | Categorical features |
| flightType, agency | Service choices | Categorical features |
| time, distance | Journey properties | Numeric features |
| date | Booking date | Calendar-feature source |
| travelCode, userCode | Identifiers | Excluded to avoid memorisation |

### Check Unique Values for each variable.

In [ ]:
display(flights.nunique().sort_values(ascending=False).to_frame('unique_values'))

## ***3. Data Wrangling***

### Data Wrangling Code

In [ ]:
model_df = flights.dropna(subset=["date", "price", "time", "distance"]).copy()
model_df["route"] = model_df["from"].astype(str) + " → " + model_df["to"].astype(str)
model_df["year"] = model_df["date"].dt.year
model_df["month"] = model_df["date"].dt.month
model_df["day"] = model_df["date"].dt.day
model_df["dayofweek"] = model_df["date"].dt.dayofweek
model_df["quarter"] = model_df["date"].dt.quarter
model_df["is_weekend"] = (model_df["dayofweek"] >= 5).astype(int)
model_df = model_df.sort_values("date").reset_index(drop=True)
print("Rows retained:", len(model_df))

### What all manipulations have you done and insights you found?

Rows missing a price, date, time, or distance cannot form a reliable supervised
example, so they are removed. Date is expanded into calendar features and origin
plus destination is combined into a route. This enables the model to learn
seasonality and route-specific pricing without using record identifiers.

## ***4. Data Visualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1: Flight-price distribution

##### 1. Why did you pick the specific chart?
A histogram reveals target spread and skew.

##### 2. What is/are the insight(s) found from the chart?
It shows the typical price range and any long tail.

##### 3. Will the gained insights help creating a positive business impact?
It makes later dollar-error metrics easier to interpret.

In [ ]:
sns.histplot(data=model_df, x='price', bins=50, kde=True); plt.title('Flight-price distribution'); plt.show()

#### Chart - 2: Flight-duration distribution

##### 1. Why did you pick the specific chart?
Duration is a direct numeric model input.

##### 2. What is/are the insight(s) found from the chart?
It exposes usual versus unusually long journeys.

##### 3. Will the gained insights help creating a positive business impact?
It validates duration before using it in price estimation.

In [ ]:
sns.histplot(data=model_df, x='time', bins=50); plt.title('Flight-duration distribution'); plt.show()

#### Chart - 3: Distance distribution

##### 1. Why did you pick the specific chart?
Distance is expected to be an important cost driver.

##### 2. What is/are the insight(s) found from the chart?
It displays the range of route lengths.

##### 3. Will the gained insights help creating a positive business impact?
It supports comparing local and long-haul flight behaviour.

In [ ]:
sns.histplot(data=model_df, x='distance', bins=50); plt.title('Flight-distance distribution'); plt.show()

#### Chart - 4: Bookings by ticket type

##### 1. Why did you pick the specific chart?
A count chart checks class representation.

##### 2. What is/are the insight(s) found from the chart?
It reveals whether one service type dominates historical records.

##### 3. Will the gained insights help creating a positive business impact?
It identifies classes where future data coverage should be monitored.

In [ ]:
sns.countplot(data=model_df, x='flightType'); plt.xticks(rotation=30); plt.title('Bookings by ticket type'); plt.show()

#### Chart - 5: Price by ticket type

##### 1. Why did you pick the specific chart?
A boxplot compares price distributions across classes.

##### 2. What is/are the insight(s) found from the chart?
It shows central price differences and within-class spread.

##### 3. Will the gained insights help creating a positive business impact?
It supports keeping ticket type as a predictive feature.

In [ ]:
sns.boxplot(data=model_df, x='flightType', y='price'); plt.xticks(rotation=30); plt.title('Price by ticket type'); plt.show()

#### Chart - 6: Price versus distance

##### 1. Why did you pick the specific chart?
A sampled scatterplot shows a relationship without overplotting.

##### 2. What is/are the insight(s) found from the chart?
It shows that distance matters but does not alone fix a price.

##### 3. Will the gained insights help creating a positive business impact?
It supports combining distance with route, agency, class, and calendar fields.

In [ ]:
sample = model_df.sample(min(8000, len(model_df)), random_state=RANDOM_STATE); sns.scatterplot(data=sample, x='distance', y='price', alpha=.25); plt.title('Price versus distance'); plt.show()

#### Chart - 7: Monthly average price

##### 1. Why did you pick the specific chart?
A line chart tests for seasonal variation.

##### 2. What is/are the insight(s) found from the chart?
It displays average price changes across booking months.

##### 3. Will the gained insights help creating a positive business impact?
It justifies date-derived features for quote-time prediction.

In [ ]:
monthly = model_df.groupby('month', as_index=False)['price'].mean(); sns.lineplot(data=monthly, x='month', y='price', marker='o'); plt.title('Average price by month'); plt.show()

#### Chart - 8: Highest-priced routes

##### 1. Why did you pick the specific chart?
Route is a business-readable combination of origin and destination.

##### 2. What is/are the insight(s) found from the chart?
It identifies historically expensive routes with sufficient data.

##### 3. Will the gained insights help creating a positive business impact?
It supports budget conversations and route-specific monitoring.

In [ ]:
route_mean = model_df.groupby('route').filter(lambda x: len(x) >= 20).groupby('route')['price'].mean().nlargest(12).sort_values(); route_mean.plot(kind='barh', title='Highest-priced routes'); plt.show()

#### Chart - 9: Correlation heatmap

##### 1. Why did you pick the specific chart?
A heatmap summarizes numeric associations.

##### 2. What is/are the insight(s) found from the chart?
It reveals relationships between price, distance, time, and calendar fields.

##### 3. Will the gained insights help creating a positive business impact?
It supports defensible feature selection and troubleshooting.

In [ ]:
cols = ['price','time','distance','year','month','day','dayofweek','quarter','is_weekend']; sns.heatmap(model_df[cols].corr(), cmap='coolwarm', center=0); plt.title('Numeric correlation heatmap'); plt.show()

#### Chart - 10: Bookings by agency

##### 1. Why did you pick the specific chart?
Agency representation affects model coverage.

##### 2. What is/are the insight(s) found from the chart?
It shows the highest-volume agencies.

##### 3. Will the gained insights help creating a positive business impact?
It highlights categories that may need more data before a business rollout.

In [ ]:
top = model_df['agency'].value_counts().head(12).index; sns.countplot(data=model_df[model_df['agency'].isin(top)], y='agency', order=top); plt.title('Top agencies'); plt.show()

## **5. Solution to Business Objective**

A fitted pipeline can score a new quote in real time. The price estimate is a decision-support input, not a guarantee: monitoring later booking data is necessary before setting any customer-facing commitment.

## ***6. Feature Engineering & Data Pre-processing***

Categorical fields are one-hot encoded and numeric fields pass through unchanged.
Both steps are placed inside a Pipeline, which ensures that the transformations
learned from the training set are applied identically to unseen data.

## ***7. Model Implementation***

### Histogram Gradient Boosting Regressor

In [ ]:
numeric = ['time', 'distance', 'year', 'month', 'day', 'dayofweek', 'quarter', 'is_weekend']
categorical = ['from', 'to', 'flightType', 'agency', 'route']
X = model_df[numeric + categorical]
y = model_df['price'].astype(float)
split_at = int(len(X) * .80)
X_train, X_test = X.iloc[:split_at], X.iloc[split_at:]
y_train, y_test = y.iloc[:split_at], y.iloc[split_at:]

preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical),
    ('numeric', 'passthrough', numeric),
], sparse_threshold=0)
flight_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', HistGradientBoostingRegressor(learning_rate=.08, max_leaf_nodes=63,
                                                 max_iter=180, l2_regularization=1.0,
                                                 random_state=RANDOM_STATE)),
])
flight_model.fit(X_train, y_train)
predicted_price = flight_model.predict(X_test)

## ***8. Model Performance & Evaluation***

MAE is the average absolute dollar error. RMSE penalises larger errors more
strongly, and R² describes the variance explained by the features. The test set is
the most recent 20% of the time-ordered records, not a random slice.

In [ ]:
metrics = {
    'RMSE': float(np.sqrt(mean_squared_error(y_test, predicted_price))),
    'MAE': float(mean_absolute_error(y_test, predicted_price)),
    'R2': float(r2_score(y_test, predicted_price)),
    'train_rows': len(X_train), 'test_rows': len(X_test),
}
display(pd.Series(metrics).to_frame('value'))

### Evaluation interpretation

Interpret the result as holdout performance on the supplied data. A high R² is
encouraging, but it should be revalidated against newly arriving data because
price relationships can change with market conditions and data collection rules.

## ***9. MLOps & Deployment***

The GitHub project saves the pipeline with joblib, writes JSON metrics, tracks runs
with MLflow when configured, serves predictions with Flask, and includes Docker,
Kubernetes, Airflow, and Jenkins configurations for repeatable delivery.

# **Conclusion**

This notebook creates a time-aware flight-price baseline from quote-time features.
The next production step is monitoring price error and data drift by route, agency,
and month.

### ***Hurrah! You have successfully completed your Regression Capstone Project !!!***